In [395]:
import sys
sys.path.append("../eng-quant/")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [396]:
from ipm.ipm_solve_no_split_so import ipm_solve_no_split_so # type: ignore[reportMissingImports]
from relaxers.remap_lower_bounds import remap_lower_bounds # type: ignore[reportMissingImports]
from relaxers.quadratic_model_lin_approx import quadratic_model_lin_approx # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp_return_x # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np

In [397]:
LIN_APPROX = 12

# Whether to do an inner or outer approximation
OUTER_APPROXIMATION = True

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

f_name = "GasLib-40_reform.json"

In [398]:
quad, q_time, q_x = solve_qcp_return_x(f_name)
print(quad)

Set parameter BarConvTol to value 1e-12
Set parameter BarQCPConvTol to value 1e-12
Set parameter NLBarCFeasTol to value 1e-12
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 7 8845HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
BarConvTol  1e-12
BarQCPConvTol  1e-12
NLBarCFeasTol  1e-12

Presolve a model with 0 rows, 85 columns and 0 nonzeros (Min)
Model fingerprint: 0x6e2f7714
Model has 23 linear objective coefficients
Model has 130 quadratic constraints
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  QMatrix range    [3e-05, 4e-02]
  QLMatrix range   [1e+00, 2e+00]
  Objective range  [1e+00, 3e+00]
  Bounds range     [1e+00, 1e+04]
  RHS range        [0e+00, 0e+00]
  QRHS range       [8e+01, 5e+03]
Presolve removed 0 rows and 14 columns
Statistics for model 'qcp_from_json_pre':
  Problem type    

In [399]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                            f_name=f_name)
remap_lower_bounds(f_name="linear_approx.json")
val, time, x, compl = ipm_solve_no_split_so()
print(x)

# print(condition_number_nes_basic())
print(f"Error: {val - quad}")
prev_err = val - quad

Time to load matrices: 0.01 seconds.
Solution is below target precision.
The algorithm stopped after 36 iterations in 0.05 seconds.

Primal objective:        -4.02398198e+03
Dual objective:          -4.02398198e+03

Norm of Primal residual: 5.68e-15
Norm of Dual residual:   6.17e-10
Abs. Compl. Gap:         9.82e-09
Rel. Compl. Gap:         1.22e-12

[7.25000000e+02 7.50000000e+01 2.00000000e+02 2.75000000e+02 3.50000000e+02 6.21215225e+02 4.19449651e+02 3.44449651e+02 1.26765574e+02 1.81649119e+02 2.69449651e+02 5.75000000e+02 4.44496511e+01
 1.50000000e+02 2.25000000e+02 7.50000000e+01 1.50000000e+02 7.50000000e+01 2.55550349e+02 3.30550349e+02 2.33414693e+02 1.72135656e+02 7.50000000e+01 2.47135656e+02 5.03784775e+02 4.25000000e+02
 2.88301806e+02 7.50000000e+01 2.86698194e+02 2.11698194e+02 7.25000000e+02 3.99324817e+02 3.99943187e-11 3.25675183e+02 3.25675183e+02 2.50675183e+02 1.50000000e+02 5.75000000e+02 3.99943187e-11
 1.02000000e+04 1.00750000e+04 1.05750000e+04 1.07250000e+0

In [400]:
# Define a set of functions for each

# quads = [[val] for val in x[:10]]
# quads = [[val] for val in x[:23]]
quads = [[val] for val in x[:39]]
# print(quads)
iters = 30
precision = 1e-4

for i in range(1, iters + 1):
    print(f"Iteration {i}")
    points_functions = []

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
            uniform = np.linspace(lower, upper, LIN_APPROX)
            for q in qs:
                idx = np.searchsorted(uniform, q)
                uniform = np.insert(uniform, idx, q)
            while len(uniform) > num:
                uniform = uniform[1:]
                if len(uniform) > num:
                    uniform = uniform[:-1]
            return uniform
        return np_uniform_add_qs

    for qs in quads:
        points_functions.append(np_uniform_add_q_factory(qs))

    # Generally, eps must be set higher for inner approx and lower for outer approx
    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name=f_name, eps=1e-2)
    remap_lower_bounds(f_name="linear_approx.json")

    val, time, x, compl = ipm_solve_no_split_so(f_name="linear_approx.json")

    # print(f"x: {x}")
    if np.abs(val - quad) > np.abs(prev_err):
        print(f"Worse Result on iteration {i}: {val - quad}")
        break
    else:
        prev_err = val - quad
    print(f"val: {val}")
    print(f"Error: {val - quad}")

    # Precision
    if abs(prev_err) < precision:
        print("Solution below precision; breaking")
        break

    for j, qs in enumerate(quads):
        qs.append(x[j])

    print(quads)
    # print(compl)

# print(x)
print(f"Final Error: {prev_err}")

Iteration 1
Time to load matrices: 0.01 seconds.
The solution quality is limited by the precision of the linear system solver.
The algorithm stopped after 59 iterations in 0.07 seconds.

Primal objective:        -1.06211286e+04
Dual objective:          -1.06211286e+04

Norm of Primal residual: 8.21e-15
Norm of Dual residual:   5.10e-10
Abs. Compl. Gap:         1.11e-08
Rel. Compl. Gap:         5.22e-13

val: 10621.128561091584
Error: -214.5113536067838
[[np.float64(724.9999999999997), np.float64(724.999999999998)], [np.float64(75.0), np.float64(75.0)], [np.float64(199.99999999999983), np.float64(200.0000000000017)], [np.float64(274.99999999999983), np.float64(275.00000000000165)], [np.float64(349.99999999999983), np.float64(350.00000000000165)], [np.float64(621.2152248620074), np.float64(717.3705408802089)], [np.float64(419.44965106116933), np.float64(485.59089492160797)], [np.float64(344.44965106116933), np.float64(410.59089492160797)], [np.float64(126.76557380083807), np.float64(156.